# Datalab 2 - Sprint 2
## groep 6 : 
- Ardjun Debi - Tewarie (25032895)
- Azfar Mohab Ali (25026356)
- Tesja Kartomo (25035193)
- Shelby Gibbs (18005055)

## Import en connectie met de database

Hier maken we gebruik van een `DatabaseViewer` class die de verbinding met de database (`./assets/database.sqlite`) opzet en functies bevat om SQL queries uit te voeren en zowel volledige resultaten als een preview (eerste rij) weer te geven.

In [1]:
import sqlite3
import pandas as pd

class DatabaseViewer:
    """
    Klasse voor het verbinden met en bekijken van data uit een SQLite database.

    Attributes
    ----------
    db_path : str
        Pad naar de database.
    conn : sqlite3.Connection
        Actieve databaseverbinding.
    """

    def __init__(self, db_path="./assets/database.sqlite"):
        """
        Initialiseert de databaseverbinding.

        Parameters
        ----------
        db_path : str, optional
            Pad naar de SQLite database (standaard './assets/database.sqlite').
        """
        self.db_path = db_path
        self.conn = sqlite3.connect(db_path)
        print(f"Database connectie is gemaakt met: {db_path}")

    def show_query(self, query):
        """
        Voert een SQL-query uit.

        Parameters
        ----------
        query : str
            De SQL-query die uitgevoerd moet worden.

        Returns
        -------
        Datafframe
            Toont het resultaat als pandas DataFrame.
        """
        df = pd.read_sql_query(query, self.conn)
        display(df)

    def show_head_query(self, query):
        """
        Voert een SQL-query uit en toont de eerste rij.

        Parameters
        ----------
        query : str
            De SQL-query die uitgevoerd moet worden.

        Returns
        -------
        Eerste rij dataframe
            Toont alleen de eerste rij van het resultaat.
        """
        df = pd.read_sql_query(query, self.conn)
        display(df.head(1))


bar = DatabaseViewer()


Database connectie is gemaakt met: ./assets/database.sqlite


## Stap 0: Structuur van de database bekijken

- **Stap 0.1:** Bekijk alle tabellen in de database.  
- **Stap 0.2 – 0.8:** Bekijk de kolommen van de belangrijkste tabellen (`Player_Attributes`, `Player`, `Match`, `League`, `Country`, `Team`, `Team_Attributes`) om te begrijpen welke data beschikbaar is.

In [2]:
#stap 0: Kijk naar de structuur van de db
#stap 0.1: alle tabellen uit de database
print("All tabellen uit de database:")
bar.show_query(
    """
    SELECT name as tabelnamen
    FROM sqlite_master
    WHERE type='table';
    """
)
#stap 0.2: alle kolommen uit de tabel: Player_Attributes
print("alle kolommen uit de tabel: Player_Attributes:")
bar.show_head_query(
    """
    SELECT * from Player_Attributes
    """
)
#stap 0.3: alle kolommen uit de tabel: Player
print("alle kolommen uit de tabel: Player:")
bar.show_head_query(
    """
    SELECT * from Player
    """
)
#stap 0.4: alle kolommen uit de tabel Match
print("alle kolommen uit de tabel: Match:")
bar.show_head_query(
    """
    SELECT * from Match
    """
)
#stap 0.5: alle kolommen uit de tabel: League
print("alle kolommen uit de tabel: League:")
bar.show_head_query(
    """
    SELECT * from League
    """
)
#stap 0.6 alle kolommen uit de tabel: Country
print("alle kolommen uit de tabel: Country:")
bar.show_head_query(
    """
    SELECT * from Country
    """
)
#stap 0.7: alle kolommen uit de tabel: Team
print("alle kolommen uit de tabel: Team:")
bar.show_head_query(
    """
    SELECT * from Team
    """
)

#stap 0.8: alle kolommen uit de tabel: Team_Attributes
print("alle kolommen uit de tabel: Team_Attributes:")
bar.show_head_query(
    """
    SELECT * from Team_Attributes
    """
)

All tabellen uit de database:


DatabaseError: Execution failed on sql '
    SELECT name as tabelnamen
    FROM sqlite_master
    WHERE type='table';
    ': file is not a database

## Stap 1: team_api_id van Barcelona vinden

- **Stap 1:** Zoek in de tabel **Team** naar de `team_api_id` van **FC Barcelona**. Deze identifier gebruiken we later om wedstrijden van dit team te vinden.

In [ ]:
# stap 1: vind de team_api_id van FC Barcelona
bar.show_query("""
SELECT team_long_name, team_api_id
FROM Team
WHERE team_long_name = 'FC Barcelona';
""")


,team_long_name,team_api_id
0,FC Barcelona,8634


## Stap 2: Matches van Barcelona vinden

- **Stap 2:** Zoekt naar wedstrijden waarin Barcelona speelt door `team_api_id` (8634) in de tabel **Match** te filteren. Hierbij kijken we zowel naar `home_team_api_id` als `away_team_api_id`.

In [ ]:
# stap 2: vind de matches door middel van team_api_id van barcelona
bar.show_query("""
SELECT
  Match.id,
  Match.season,
  Match.country_id,
  Match.league_id,
  Match.home_team_api_id,
  Match.away_team_api_id
FROM Match
WHERE Match.home_team_api_id = 8634
   OR Match.away_team_api_id = 8634
LIMIT 5;
          
""")

,id,season,country_id,league_id,home_team_api_id,away_team_api_id
0,21521,2008/2009,21518,21518,8388,8634
1,21535,2008/2009,21518,21518,8634,10281
2,21547,2008/2009,21518,21518,8479,8634
3,21550,2008/2009,21518,21518,8634,8305
4,21564,2008/2009,21518,21518,8302,8634


## Stap 3: League van Barcelona bepalen

- **Stap 3.1:** Zoek de `league_id` van een wedstrijd waarin Barcelona speelt.  
- **Stap 3.2:** Zoek de `country_id` van deze wedstrijd.  
- **Stap 3.3:** Gebruik de `country_id` om de naam van de **League** te vinden.

In [ ]:
# stap 3: vind de league_id's en de naam van de league van de matches van barcelona
# stap 3.1: vind de league id
bar.show_query(
"""
SELECT Match.league_id
FROM Match
WHERE Match.home_team_api_id = 8634 OR Match.away_team_api_id = 8634
LIMIT 1;
""")

# stap 3.2: vind eerste de country id om makkelijjker te zoeken naar de name van de League
bar.show_query(
"""
SELECT Match.country_id
FROM Match
WHERE home_team_api_id = 8634
LIMIT 1
"""
)
# stap 3.3 vind nu de naam van de league:
bar.show_query(
"""
SELECT League.Name
FROM League
WHERE country_id = 21518
LIMIT 1
"""
)



,league_id
0,21518


,country_id
0,21518


,name
0,Spain LIGA BBVA


## Belangrijke variablen
- Team : **Barcelona**
- `Team_api_id`, `home_team_api_id`, `away_team_api_id` : 8634
- `Country_id`, `League_id` : 21518
- `League.Name` : Spain LIGA BBVA

## Relaties:
### Tabel met relaties:
| Kolom| Van tabel | Naar tabel | Naar kolom | Beschrijving|
| ------------------ | ----------------- | ---------- | ------------------ | ----------------------------------------------------------------------- |
| country_id | League| Country| id | Geeft aan in welk land de **competitie plaatsvindt**. |
| country_id | Match | Country| id | Geeft aan in welk land de **wedstrijd gespeeld** wordt. |
| league_id| Match | League | id | **Verbindt** een wedstrijd aan de competitie waarin deze gespeeld wordt.|
| home_team_api_id | Match | Team | team_api_id| Verwijst naar het team dat de **thuiswedstrijd** speelt.|
| away_team_api_id | Match | Team | team_api_id| Verwijst naar het team dat de **uitwedstrijd** speelt.|
| player_api_id| Player_Attributes | Player | player_api_id| **Verbindt** de spelers attributen met een specifieke speler.|
| player_fifa_api_id | Player_Attributes | Player | player_fifa_api_id | Alternatieve identifier uit FIFA dat die attributen aan een speler **koppelt**. |
| team_api_id| Team_Attributes | Team | team_api_id| **Verbindt** team attributen met een specifiek team. |
| team_fifa_api_id | Team_Attributes | Team | team_fifa_api_id | FIFA identifier die team attributen **koppelt** aan een team.|
| away_player_1| Match | Player | player_api_id| Eerste speler van het **uit team** in de wedstrijd. |
| away_player_2| Match | Player | player_api_id| Tweede speler van het **uit team** in de wedstrijd. |
| away_player_3| Match | Player | player_api_id| Derde speler van het **uit team** in de wedstrijd.|
| away_player_4| Match | Player | player_api_id| Vierde speler van het **uit team** in de wedstrijd. |
| away_player_5| Match | Player | player_api_id| Vijfde speler van het **uit team** in de wedstrijd. |
| away_player_6| Match | Player | player_api_id| Zesde speler van het **uit team** in de wedstrijd.|
| away_player_7| Match | Player | player_api_id| Zevende speler van het **uit team** in de wedstrijd.|
| away_player_8| Match | Player | player_api_id| Achtste speler van het **uit team** in de wedstrijd.|
| away_player_9| Match | Player | player_api_id| Negende speler van het **uit team** in de wedstrijd.|
| away_player_10 | Match | Player | player_api_id| Tiende speler van het **uit team** in de wedstrijd. |
| away_player_11 | Match | Player | player_api_id| Elfde speler van het **uit team** in de wedstrijd.|
| home_player_1| Match | Player | player_api_id| Eerste speler van het **thuis team** in de wedstrijd. |
| home_player_2| Match | Player | player_api_id| Tweede speler van het **thuis team** in de wedstrijd. |
| home_player_3| Match | Player | player_api_id| Derde speler van het **thuis team** in de wedstrijd.|
| home_player_4| Match | Player | player_api_id| Vierde speler van het **thuis team** in de wedstrijd. |
| home_player_5| Match | Player | player_api_id| Vijfde speler van het **thuis team** in de wedstrijd. |
| home_player_6| Match | Player | player_api_id| Zesde speler van het **thuis team** in de wedstrijd.|
| home_player_7| Match | Player | player_api_id| Zevende speler van het **thuis team** in de wedstrijd.|
| home_player_8| Match | Player | player_api_id| Achtste speler van het **thuis team** in de wedstrijd.|
| home_player_9| Match | Player | player_api_id| Negende speler van het **thuis team** in de wedstrijd.|
| home_player_10 | Match | Player | player_api_id| Tiende speler van het **thuis team** in de wedstrijd. |
| home_player_11 | Match | Player | player_api_id| Elfde speler van het **thuis team** in de wedstrijd.|

### Samengevat:
Elke wedstrijd is gekoppeld aan een competitie via `league_id` en aan twee teams via `home_team_api_id` en `away_team_api_id`, die verwijzen naar de tabel **Team**.

De spelers die in een wedstrijd spelen staan in de kolommen `home_player_1` t/m `home_player_11` en `away_player_1` t/m `away_player_11`. Deze verwijzen naar de tabel **Player**.

De tabellen `Player_Attributes` en `Team_Attributes` bevatten extra informatie over **spelers** en **teams** en zijn gekoppeld via hun **API identifiers**.


## Ranglijst
Seizoen : 2015

In [ ]:
bar.show_query(
"""
SELECT date
FROM match 
WHERE date LIKE "2015%"
""")

In [ ]:
# Ranglijst - La Liga seizoen 2015/2016
import sqlite3
import pandas as pd

# Verbinding maken met de database
conn = sqlite3.connect("./assets/database.sqlite")


def get_ranglijst(conn, league_id, seizoen):
    """
    Berekent de ranglijst van een competitie voor een gegeven seizoen.

    Parameters:
        conn       : sqlite3 verbinding met de database
        league_id  : int, de league_id van de competitie (bijv. 21518 voor La Liga)
        seizoen    : str, het seizoen als string (bijv. '2015/2016')

    Returns:
        pd.DataFrame met kolommen: Positie, Team, GS, W, G, V, DV, DT, DS, Punten
        gesorteerd op punten (desc), dan doelsaldo (desc), dan doelpunten voor (desc)
    """

    # Stap 1: Haal alle wedstrijden op uit het gekozen seizoen en competitie
    query = f"""
        SELECT
            m.home_team_api_id AS home_id,
            m.away_team_api_id AS away_id,
            m.home_team_goal   AS home_goal,
            m.away_team_goal   AS away_goal
        FROM Match m
        WHERE m.league_id = {league_id}
          AND m.season = '{seizoen}'
    """
    matches = pd.read_sql_query(query, conn)

    # Stap 2: Haal de teamnamen op zodat we ID's kunnen omzetten naar namen
    team_query = "SELECT team_api_id, team_long_name FROM Team"
    teams = pd.read_sql_query(team_query, conn)
    team_dict = dict(zip(teams['team_api_id'], teams['team_long_name']))

    # Stap 3: Maak een lege statistieken-dictionary aan voor elk uniek team
    alle_teams = set(matches['home_id']).union(set(matches['away_id']))
    stats = {t: {'GS': 0, 'W': 0, 'G': 0, 'V': 0, 'DV': 0, 'DT': 0} for t in alle_teams}

    # Stap 4: Loop door alle wedstrijden en bereken de statistieken per team
    for _, row in matches.iterrows():
        h, a   = row['home_id'],   row['away_id']
        hg, ag = row['home_goal'], row['away_goal']

        # Aantal gespeelde wedstrijden ophogen voor beide teams
        stats[h]['GS'] += 1
        stats[a]['GS'] += 1

        # Doelpunten bijhouden voor beide teams
        stats[h]['DV'] += hg
        stats[h]['DT'] += ag
        stats[a]['DV'] += ag
        stats[a]['DT'] += hg

        # Bepaal de uitslag winst/gelijk/verlies
        if hg > ag:        # Thuisploeg wint
            stats[h]['W'] += 1
            stats[a]['V'] += 1
        elif hg < ag:      # Uitploeg wint
            stats[a]['W'] += 1
            stats[h]['V'] += 1
        else:              # Gelijkspel
            stats[h]['G'] += 1
            stats[a]['G'] += 1

    # Stap 5: Zet de statistieken om naar een overzichtelijke lijst
    rijen = []
    for team_id, s in stats.items():
        punten = s['W'] * 3 + s['G']   # Winst = 3 punten, gelijk = 1 punt
        ds     = s['DV'] - s['DT']     # Doelsaldo = doelpunten voor - tegen

        rijen.append({
            'Team'   : team_dict.get(team_id, str(team_id)),
            'GS'     : s['GS'],
            'W'      : s['W'],
            'G'      : s['G'],
            'V'      : s['V'],
            'DV'     : s['DV'],
            'DT'     : s['DT'],
            'DS'     : ds,
            'Punten' : punten
        })

    # Stap 6: Maak een DataFrame en sorteer op punten, doelsaldo, doelpunten voor
    ranglijst = pd.DataFrame(rijen)
    ranglijst = ranglijst.sort_values(
        by=['Punten', 'DS', 'DV'],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    # Positie begint bij 1 in plaats van 0
    ranglijst.index += 1
    ranglijst.index.name = 'Positie'

    return ranglijst


# Roep de functie aan voor La Liga seizoen 2015/2016
ranglijst_2015 = get_ranglijst(conn, league_id=21518, seizoen='2015/2016')
display(ranglijst_2015)

## Uitleg van de ranglijst

De ranglijst laat zien hoe alle clubs in La Liga het hebben gedaan in het seizoen 2015/2016. Hieronder leggen we uit wat elke kolom betekent:

- **Positie** – De plek van het team in de competitie, gebaseerd op het aantal punten.
- **Team** – De naam van de voetbalclub.
- **GS** – Gespeelde wedstrijden. Hoeveel wedstrijden het team heeft gespeeld dat seizoen.
- **W** – Gewonnen. Het aantal wedstrijden dat het team heeft gewonnen.
- **G** – Gelijk. Het aantal wedstrijden dat gelijkgespeeld is.
- **V** – Verloren. Het aantal wedstrijden dat het team heeft verloren.
- **DV** – Doelpunten Vóór. Het totaal aantal doelpunten dat het team zelf heeft gescoord.
- **DT** – Doelpunten Tégen. Het totaal aantal doelpunten dat het team heeft weggegeven.
- **DS** – Doelsaldo. Het verschil tussen DV en DT (DV - DT). Een positief getal betekent dat een team meer heeft gescoord dan weggegeven.
- **Punten** – Het totaal aantal punten. Een overwinning levert 3 punten op, een gelijkspel 1 punt en een verlies 0 punten. Bij een gelijk aantal punten wordt gekeken naar het doelsaldo.

------------------------------------------------------------------------------------------------------------------------

# SPRINT 3
## 1A: Aantal wedstrijden per seizoen
Hier hebben we met behulp van SQL het aantal wedstrijden per seizoen van FC Barcelona kunnen aan tonen.


Stap 1: Tel, met gebruik van `league_id`, `home_team_api_id` en `away_team_api_id`, de aantal wedstrijden per seizoen.

In [ ]:
#Stap 1: Toon het aantal wedstrijden per seizoen
bar.show_query(
"""
SELECT league.name, season, COUNT(*) AS aantal_wedstrijden
FROM match
INNER JOIN league ON match.league_id = league.id
WHERE home_team_api_id = 8634 
   OR away_team_api_id = 8634
GROUP BY league.name, season;
"""
)

### 1B: Aantal wedstrijden in het kalenderjaar 2010 per seizoen


In [ ]:
#Stap 2: toon het aantal wedstrijden per seizon in het kalenderjaar 2010\
bar.show_query(
"""
SELECT league.name, season, COUNT(*) AS aantal_wedstrijden
FROM match
INNER JOIN league ON match.league_id = league.id
WHERE (home_team_api_id = 8634 OR away_team_api_id = 8634)
  AND season LIKE '2010%'
GROUP BY league.name, season;
"""
)



###  1C: Hoeveelheid punten ieder team had gehaald per seizoen 

In [ ]:
#Stap 3: toon hoeveel punten ieder team in ons compititie gehaald heeft per seizoen
bar.show_query(
"""SELECT
    league.name AS league,
    season,
    team.team_long_name AS team,
    SUM(
        CASE 
            WHEN match.home_team_api_id = team.team_api_id AND home_team_goal > away_team_goal THEN 3
            WHEN match.away_team_api_id = team.team_api_id AND away_team_goal > home_team_goal THEN 3
            WHEN home_team_goal = away_team_goal THEN 1
            ELSE 0 
        END
    ) AS punten
FROM match
INNER JOIN league ON match.league_id = league.id
INNER JOIN team ON team.team_api_id = match.home_team_api_id 
               OR team.team_api_id = match.away_team_api_id
WHERE league.name = 'Spain LIGA BBVA'
GROUP BY league.name, season, team.team_api_id
ORDER BY season, punten DESC;
"""
)

# case doet hier dus het volgende: als het team de thuisploeg is en wint, krijgt het 3 punten. Als het team de uitploeg is en wint, krijgt het ook 3 punten. Als er een gelijkspel is, krijgen beide teams 1 punt. In alle andere gevallen (verlies) krijgt het team 0 punten.



### 1D: Rank tonen op ranglijst


In [ ]:
#Stap 3: Toon op welke plaats Barcelona is geëindigd in de competitie per seizoen
bar.show_query
